# U5-L4：语音问答闭环

**课程目标**

1. 能说出一次语音问答里，从开口到听见回答要走过哪几段，以及每一段大概在做什么。
2. 能画出一个对话状态机的状态与转换条件：待机、聆听、处理、回答，各自什么时候进、什么时候出。
3. 能测出一次完整问答的用时，并指出这些时间主要花在哪一段。


## 一、本课要解决的问题

前三课各做了一段：把话变成字、把字变成声音、如何理解并回答问题。

一次真实的聊天问答，不会像前三课那样一段一段分着跑。人问一句，机器人得自己接着往下走：先听清，再想，再回答，然后接着等下一句。

本课任务要求是：对着机器人问一个问题，它按设定的人设回答，整段过程可以连续跑。

> **重点思考：从我说完到机器人的第一句话，这几秒钟里都发生了什么？**


## 二、一次问答要走过哪几段

一次问答是一条流水线：都有哪些段？时间又花在哪？

### 2.1 四段链路

一次问答走过四段：

- **听见**：麦克风把声音变成采样点；
- **听懂**：把声音里的那句话变成文字（ASR）；
- **想**：按设定和上下文，接出要说的内容（LLM）；
- **说**：把内容变成声音放出来（TTS）。

四段是从前往后顺序执行的，前一段的输出就是后一段的输入。中间哪一段卡住，后面全都不会发生。

前面三课分别在里面做过事：
- L1 练的是「听见 → 听懂」
- L2 练的是「说」
- L3 管的是「想」那一段按什么规矩来。

### 2.2 每一段谁在干活

四段并不都由同一方负责。分清这一点，出问题的时候才知道往哪儿找。

- 听见：机器人本体（麦克风）；
- 听懂：模型服务，跑在机器人之外；
- 想：模型服务，也在机器人之外；
- 说：模型服务合成，机器人本体放出来（扬声器）。

其中多段涉及“服务”，所以需要网络。网络一慢，对应的段的用时就会跟着涨。

程序这边管的是「什么时候发起、发起了之后怎么等」。这件事正好就是这一课要学习的状态机。

### 2.3 延迟花在哪

一次问答的**端到端延迟**，是从人说完最后一个字，到听见机器人第一个字的这段时间。

它的组成大致是三块：声音上传、模型回复、回复合成并下载回来。三块的快慢都跟网络和服务的忙闲有关，同一天不同时段测出来的数可能不一样。

L1 里，队列的深浅决定声音落后多久；这三块，每一块都要花自己的时间——加起来，人就觉得机器人在发愣。

### 2.4 人能忍多久

延迟合不合格，先要看人的感受。

大体上：一秒以内，人觉得是「立刻就答」；两三秒，还能接受，但会觉得对方想了一下；五秒往上，人会怀疑是不是坏了，开始重复说话或者去按键。

所以展厅这类场景里，常见做法是先把一段完整交流的总时间控制在三四秒。要控制时间，办法大致就这几种：让回复短一点、少绕一趟网络、在等待的时候给个提示（比如动作），让人知道它在干活。

当然，还有一个更好的思路，就是让模型运行在机器人身体内，这是另一项有挑战的任务，本门课程暂时不做深入探讨，感兴趣的学生可以自行探索。

## 三、对话是有状态的

模型再快，对话也得有流程管着。

### 3.1 为什么要状态

对话里最怕的是抢话：它还在说，麦克风开始采集到其它的声音。

要避开抢话，程序得知道「现在轮到谁」。用一个变量就能记住：现在是什么状态。

**状态机**就是干这个的：把流程切成有限几个状态，每个状态只说清两件事——进来的时候做什么、什么条件下换到下一个。

### 3.2 四个状态

这一课用四个状态，四个状态跑完一圈会回到起点：

| 状态 | 这一段在做什么 |
| :--- | :--- |
| 待机 | 什么都不做，等一个开始的信号 |
| 聆听 | 麦克风收音，识别在出字 |
| 处理 | 把听到的这句话交给模型，等回复 |
| 回答 | 合成并播出来，说的时候不听 |

跑完「回答」回到「待机」，就接上了下一轮。

通过设定四个状态来遵守一条规矩：**听的时候不说，说的时候不听。**

### 3.3 每个转换由什么触发

每个状态之间的切换上都有一个条件，不然程序会卡在某个状态里出不来：

- 待机 → 聆听：收到开始信号（这一课是学生按运行）；
- 聆听 → 处理：听清了，拿到了识别结果；
- 聆听 → 待机：超时还没听清，退回重来；
- 处理 → 回答：回复到了。「想」和「说」都在同一个 `chat()` 任务里，本课分不出中间那一步，只能等它整段结束；
- 回答 → 待机：放完了。

第三条是这一课要特别看的：**没听清也要有输出。** 少了这一条，程序会一直等在聆听里。

### 3.4 用程序变量写出来

状态机不需要什么库。一个字符串变量记住当前状态，循环里按状态分支，每个分支里写转换条件。

把每一步的用时也一起记下来——状态和用时放在一起打，出了问题能立刻看出卡在哪一段。

这一步写出来的东西，和 SDK 提供的 `TaskHandle` 是配合关系：状态机管「什么时候发起」，`TaskHandle` 管「发起之后它怎么了」。

## 四、实验：语音问答机器人

⚠️ **注意**：本课不让机器人走动，只用到麦克风与扬声器。机器人回答的时候不要同时说话，场地安静一些；音量先调好，中途别动。

连起来跑一遍：测量一下一轮对话的大概时间，关注各个状态的切换。

### 4.1 实验前的准备

**STEP 01：** 连接机器人，让它站好

In [ ]:
import time

from boosteros.robots.booster import BoosterRobot
from boosteros.brain import Speech
from boosteros.brain.speech import ChatConfig

robot = BoosterRobot()
speech = Speech(robot)

print("已连接：", robot.robot_info.model, robot.robot_info.serial_number)
print("当前音量：", robot.audio_manager.get_system_volume())

PROMPT = "你是一名展厅迎宾员，礼貌、克制，回答只允许一句话。"
QUESTION = "请用一句话介绍一下这个展厅。"

### 4.2 整段跑一次

先按最省事的做法跑一遍：一句话发起，剩下的交给它。

**STEP 02：** 整段跑一次，估计一下时长

<font color="red">提示：这一格会让机器人进入对话。准备好手机、秒表——从你说完最后一个字开始按，到听见机器人开口再按一次，中途不要说话。</font>

In [ ]:
print("请对着机器人问：", QUESTION)

t0 = time.time()
task = speech.chat(config=ChatConfig(system_prompt=PROMPT, volume=0.8))
print("已发起，时刻：", round(t0, 1))

time.sleep(18)
task.cancel()
print(f"这一段会话共 {time.time() - t0:.1f} 秒，状态：{task.status}")

### 4.3 把「听清」那一段单独量一次

整段用时是一个总数，看不出时间花在哪。把「听清」这一段单独拿出来量一次。

后面两段（想和说）在 SDK 里是一体的，程序只能拿到整段的用时，量不了单独的。这两段的时间要靠秒表——上一格已经这么做了。

**STEP 03：** 单独量一次听清

In [ ]:
def listen_once(seconds=6.0):
    """听一次，返回（用时，认出来的文字）"""
    heard = []

    def on_text(text):
        heard.append(text)

    t0 = time.time()
    task = speech.recognize_stream(on_result=on_text)
    time.sleep(seconds)
    task.cancel()
    return time.time() - t0, (heard[-1] if heard else "")


print("请把刚才那句话再说一遍")
t_listen, heard = listen_once()
print(f"听清这一段用了 {t_listen:.1f} 秒，认出来的是：{heard}")

### 4.4 加上状态机再跑两轮

把四段放进一个循环，每一轮打完状态再打用时。「想」和「说」都在同一次 `chat()` 里，本课拿不到「回复到了」这个信号——这一格把「处理」和「回答」并成一段来打。

没听清就退回待机，不要往下走——这是 3.3 节里写明的第三条转换。

**STEP 04：** 跑两轮，看状态和用时

<font color="red">提示：这一格会连跑两轮，共约 40 秒。每一轮开始之前看屏幕上的提示再说话，两轮之间等它说完。</font>

In [ ]:
def run_round(n, listen_seconds=6.0, answer_seconds=12.0):
    """跑一轮问答：待机 → 聆听 → 处理（含回答） → 待机，并把每一段的状态与用时打出来"""
    state = "聆听"
    print(f"第 {n} 轮　状态：{state}　请说出这句话：{QUESTION}")

    t_listen, heard = listen_once(listen_seconds)
    print(f"    听到：{heard}　听清用时 {t_listen:.1f} 秒")

    if not heard:
        print("  没听清，状态回到：待机")
        return None

    state = "处理"
    print(f"  状态：{state}")

    t1 = time.time()
    task = speech.chat(config=ChatConfig(system_prompt=PROMPT, volume=0.8))
    time.sleep(answer_seconds)
    task.cancel()
    t_answer = time.time() - t1
    print(f"    处理与回答用时 {t_answer:.1f} 秒　状态：{task.status}")

    print("  状态：待机")
    return t_listen, t_answer


for n in (1, 2):
    run_round(n)

### 4.5 异常演练

「没听清」时该怎么办，常常被漏掉。这一格专门探讨这个问题。

**STEP 05：** 分别制造一次「没听清」和一次「任务没收尾」

<font color="red">提示：第一次不要出声，等 4 秒就行。第二次照常发起对话、几秒后取消，看 `status` 和 `error` 两个字段的值——这两个字段是程序判断「它出没出事」的依据。</font>

In [ ]:
# 异常一：什么都没听清
t_listen, heard = listen_once(4.0)
print("这次没出声，拿到的是：", heard)
print("→ 程序应该退回待机重来，而不是往下走")

# 异常二：任务的状态与错误字段
task = speech.chat(config=ChatConfig(system_prompt=PROMPT, volume=0.8))
print("发起后　状态：", task.status, "　还在跑吗：", task.running())
task.cancel()
print("取消后　状态：", task.status, "　错误：", task.error)

## 五、观察与记录

### 5.1 端到端用时

| 项目 | 第 1 次 | 第 2 次 | 第 3 次 |
| :--- | :---: | :---: | :---: |
| 秒表量到的「说完 → 开口」（秒） | 　　　 | 　　　 | 　　　 |
| 整段会话用时（秒） | 　　　 | 　　　 | 　　　 |

### 5.2 各段用时

| 段 | 谁在干活 | 用时（秒） | 
| :--- | :--- | :---: |
| 听见 | 麦克风 | —— | 
| 听懂 | 模型服务 | 　　　 | 
| 想 | 模型服务 | 　　　 | 
| 说 | 模型服务合成 + 扬声器 | 　　　 | 
### 5.3 状态记录

| 轮次 | 听到的内容 | 走过了哪几个状态 | 有没有退回待机 |
| :---: | :--- | :--- | :---: |
| 第 1 轮 | 　　　　　　　　　　 | 　　　　　　　　　　 | 　　　 |
| 第 2 轮 | 　　　　　　　　　　 | 　　　　　　　　　　 | 　　　 |

### 5.4 异常记录

| 制造的异常 | 程序打印出来的状态 / 错误 | 是否与预期一致 |
| :--- | :--- | :---: |
| 没听清 | 　　　　　　　　　　　　 | 　　　 |
| 中途取消 | 　　　　　　　　　　　　 | 　　　 |

### 5.5 检查点

- 是否能说清四段链路里哪几段在机器人上、哪几段在外面。
- 是否能说出四个状态各自进来的条件和出去的条件。
- 是否知道「没听清」这条转换少了会出什么问题。

### 5.6 思考题

- 秒表量到的数和程序量到的数对不上，可能是什么原因？
- 如果把 `answer_seconds` 从 12 秒改成 6 秒，会漏掉什么？
- 机器人回答时如果同时收音，会出什么现象？如何限制？
